# C4 SE supervised simulation v1 (`se-supervised-sim-v1`)

S1 catalogue (46 topics). Semester-restricted practice + mock exams.
Additive package `training_se_supervised_v1` — does **not** modify v0.

Targets:
- **A.** Next-attempt correctness
- **B.** Synthetic mock-exam score
- **C.** Synthetic mock-exam pass probability (`PASS_THRESHOLD=0.55` labels only)

Explicitly synthetic. Not real SE / real-exam validation. Do **not** auto-wire into `se-path-sim-v1` without a new contract.

Gates default **False**. Artifacts under `se_supervised_sim_v1` only.


## 1. Dependencies


In [ ]:
%pip install -q "numpy>=1.23,<3" "pandas>=1.5,<3" "scikit-learn>=1.2,<2" "scipy>=1.10,<2" "joblib>=1.2,<2" matplotlib tqdm


## 2. Upload & extract `c4-se-supervised-v1-source-colab.zip`


In [ ]:
from pathlib import Path
import zipfile, shutil
from google.colab import files

UPLOAD_DIR = Path('/content/c4_se_sup_v1_upload')
EXTRACT_DIR = Path('/content/c4_se_sup_v1_src')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

print('Upload c4-se-supervised-v1-source-colab.zip')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded')
zip_name = next(iter(uploaded))
zip_path = UPLOAD_DIR / zip_name
zip_path.write_bytes(uploaded[zip_name])
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extracted to', EXTRACT_DIR)
print('Top:', list(EXTRACT_DIR.iterdir())[:10])


## 3. Drive mount (v1 output root only)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v1')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Artifacts root:', DRIVE_ROOT)
print('Do not write into se_supervised_sim_v0')


## 4. Imports / versions / paths


In [ ]:
import sys, os, json
from pathlib import Path

if 'EXTRACT_DIR' not in globals() or 'DRIVE_ROOT' not in globals():
    raise RuntimeError('Run cells 2 (extract) and 3 (Drive mount) first.')

hits = list(Path(EXTRACT_DIR).rglob('training_se_supervised_v1/protocol.py'))
if not hits:
    raise FileNotFoundError('training_se_supervised_v1/protocol.py not found; re-upload ZIP')
C4_ROOT = hits[0].parents[1].resolve()
sys.path.insert(0, str(C4_ROOT))
os.chdir(C4_ROOT)
print('C4_ROOT', C4_ROOT)

from training_se_supervised_v1.colab_bootstrap import require_symbol
from training_se_supervised_v1.protocol import PROTOCOL_VERSION, GENERATOR_VERSION, DOMAIN_MARKERS
from training_se_supervised_v1.provenance import capture_versions
print('protocol', PROTOCOL_VERSION, GENERATOR_VERSION)
print('markers', DOMAIN_MARKERS)
print(json.dumps(capture_versions(), indent=2))


## 5. Frozen contract + catalogue + thresholds


In [ ]:
from training_se_supervised_v1.contract import (
    frozen_contract_dict, PASS_THRESHOLD, CLASSIFICATION_PROB_THRESHOLD, SCENARIOS,
)
from training_se_supervised_v1.inventory import inventory_local_se_exports
from training_se_supervised_v1.catalogue import (
    catalogue_hash, ORDERED_TOPICS, N_TOPICS, Y3S1_TOPICS, Y3S2_TOPICS, validate_catalogue,
)
from training_se_supervised_v1.features_response import response_feature_names
from training_se_supervised_v1.features_exam import exam_feature_names

print(validate_catalogue())
print('n_topics', N_TOPICS, 'y3s1', len(Y3S1_TOPICS), 'y3s2', len(Y3S2_TOPICS))
print('response_feat_dim', len(response_feature_names()))
print('exam_feat_dim', len(exam_feature_names()))
print('PASS_THRESHOLD (synthetic label cutoff only)', PASS_THRESHOLD)
print('CLASSIFICATION_PROB_THRESHOLD (hard-class accuracy only)', CLASSIFICATION_PROB_THRESHOLD)
CONTRACT = frozen_contract_dict(mode='smoke', master_seed=42)
print('scenarios', list(SCENARIOS))
print('catalogue_sha256', catalogue_hash())
print(json.dumps(CONTRACT, indent=2)[:2800])
print(json.dumps(inventory_local_se_exports(C4_ROOT), indent=2))
(DRIVE_ROOT / '_session_contract_preview.json').write_text(
    json.dumps(CONTRACT, indent=2), encoding='utf-8'
)


## 6. Smoke generation + leakage + train/select on validation


In [ ]:
from training_se_supervised_v1.pipeline import run_training_and_validation

smoke = run_training_and_validation(
    DRIVE_ROOT,
    mode='smoke',
    master_seed=42,
    run_full=False,
    c4_root=C4_ROOT,
)
SMOKE_RUN_DIR = Path(smoke['run_dir'])
print('smoke run', SMOKE_RUN_DIR)
print('selection', json.dumps(smoke['selection'], indent=2))
print('leakage', smoke['leakage'])


## 7. Validation selection tables


In [ ]:
import pandas as pd

require_symbol('SMOKE_RUN_DIR', globals().get('SMOKE_RUN_DIR'))
sel = json.loads((SMOKE_RUN_DIR / 'selection_val.json').read_text())
resp = json.loads((SMOKE_RUN_DIR / 'comparison_val_response.json').read_text())
exam = json.loads((SMOKE_RUN_DIR / 'comparison_val_exam_score.json').read_text())

df_resp = pd.DataFrame([
    {
        'target': 'response',
        'model': r['name'],
        'val_accuracy': r['val_metrics'].get('accuracy'),
        'val_log_loss': r['val_metrics'].get('log_loss'),
        'val_brier': r['val_metrics'].get('brier'),
        'val_roc_auc': r['val_metrics'].get('roc_auc'),
        'selected': r['name'] == sel['response']['name'],
        'synthetic': True,
        'domain': 'se_simulation',
        'protocol': 'se-supervised-sim-v1',
    }
    for r in resp
])
df_exam = pd.DataFrame([
    {
        'target': 'exam_score',
        'model': r['name'],
        'val_mae': r['val_metrics'].get('mae'),
        'val_rmse': r['val_metrics'].get('rmse'),
        'selected': r['name'] == sel['exam_score']['name'],
        'synthetic': True,
        'domain': 'se_simulation',
        'protocol': 'se-supervised-sim-v1',
    }
    for r in exam
])
display(df_resp)
display(df_exam)
print('0.55 is a labelled synthetic exam-score cutoff, not an institutional rule.')
print('Do not retune generation to force 80–90% accuracy.')


## 8. Full training (gated)


In [ ]:
RUN_FULL_TRAINING = False

FULL_RUN_DIR = None
if RUN_FULL_TRAINING:
    full = run_training_and_validation(
        DRIVE_ROOT,
        mode='full',
        master_seed=42,
        run_full=True,
        c4_root=C4_ROOT,
    )
    FULL_RUN_DIR = Path(full['run_dir'])
    print('full run', FULL_RUN_DIR)
    print('selection', json.dumps(full['selection'], indent=2))
else:
    print('Skipped full training (RUN_FULL_TRAINING=False)')


## 9. Final-test evaluation (gated; frozen selection only)


In [ ]:
from training_se_supervised_v1.pipeline import run_final_evaluation, reload_run

RUN_FINAL_EVALUATION = False
FINAL_TARGET_RUN = FULL_RUN_DIR or globals().get('SMOKE_RUN_DIR')

if RUN_FINAL_EVALUATION:
    require_symbol('FINAL_TARGET_RUN', FINAL_TARGET_RUN)
    reloaded = reload_run(FINAL_TARGET_RUN)
    print('reloaded selection keys', list(reloaded['selection'].keys()))
    final = run_final_evaluation(FINAL_TARGET_RUN, run_final=True)
    print(json.dumps(final, indent=2))
else:
    print('Skipped final evaluation (RUN_FINAL_EVALUATION=False)')


## 10. Export reminder


In [ ]:
from training_se_supervised_v1.export_bundle import verify_manifest

target = globals().get('FULL_RUN_DIR') or globals().get('SMOKE_RUN_DIR')
if target is None:
    print('No run directory in this session')
else:
    export_dir = Path(target) / 'export_bundle'
    print('export_bundle', export_dir)
    print(verify_manifest(export_dir))
    print('Download export_bundle from Drive.')
    print('Marks: domain=se_simulation, synthetic=true, simulation_only=true.')
    print('Do NOT auto-wire into se-path-sim-v1 without an explicit new contract.')
